# 1. Intro to LangChain

## What is LangChain?

**LangChain** is an open-source orchestration framework for building applications on top of Large Language Models (LLMs). It is *not* an LLM itself — it is the plumbing that sits between your code and providers like OpenAI, Anthropic, Google, Groq, etc.

It gives you four core primitives that you compose together:

| Primitive | Purpose | Example class |
|---|---|---|
| **Models** | Uniform interface over chat / completion / embedding APIs | `ChatOpenAI`, `ChatAnthropic`, `OpenAIEmbeddings` |
| **Prompts** | Templated, reusable, parameterised prompts | `ChatPromptTemplate`, `PromptTemplate` |
| **Output parsers** | Coerce raw LLM text into typed Python (str, JSON, Pydantic) | `StrOutputParser`, `PydanticOutputParser` |
| **Runnables (LCEL)** | Composition layer — pipe primitives together with `\|` | `Runnable`, `RunnableLambda`, `RunnableParallel` |

On top of these primitives, the ecosystem layers higher abstractions:

- **Retrievers / Vector stores** — RAG (retrieval augmented generation)
- **Tools / Agents** — let an LLM call functions and decide control flow
- **LangGraph** — stateful, cyclic, multi-actor agent graphs
- **LangSmith** — tracing, evaluation, and observability for everything above

### Why use it (the architectural argument)

1. **Provider abstraction** — swap `ChatOpenAI` for `ChatAnthropic` without changing your business logic.
2. **Composability** — LCEL `prompt | model | parser` chains are declarative, async-native, and stream-native by default.
3. **Observability** — every Runnable invocation is automatically traced in LangSmith if `LANGSMITH_TRACING=true`.
4. **Production primitives** — retries, fallbacks, batching, streaming, caching, and parallelism are built into the Runnable interface.

### Mental model

```
input ──▶ PromptTemplate ──▶ ChatModel ──▶ OutputParser ──▶ output
              (str fmt)        (LLM call)     (str/json/obj)
```

Each `▶` is the LCEL `|` operator. Anything on either side of a `|` is a `Runnable` and exposes the same methods: `.invoke()`, `.batch()`, `.stream()`, `.ainvoke()`, `.abatch()`, `.astream()`.

## Step 1 — Load environment & verify keys

`config.environment.get_settings()` reads `AI/.env` via `pydantic-settings` and validates it. Importing `config` (the package `__init__.py`) also exports the API keys into `os.environ`, which is what the provider SDKs read internally.

In [2]:
import os
import sys
from pathlib import Path

# Make project root importable even when notebook cwd is /langchain.
project_root = Path.cwd().parent if Path.cwd().name == "langchain" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config.environment import get_settings

settings = get_settings()
settings.export_to_os_environ()

print(f"APP_ENV   : {settings.APP_ENV}")
print(f"APP_DEBUG : {settings.APP_DEBUG}")
print(f"OpenAI key loaded: {bool(os.environ.get('OPENAI_API_KEY'))}")
print(f"LangSmith tracing: {settings.LANGSMITH_TRACING}")

APP_ENV   : development
APP_DEBUG : True
OpenAI key loaded: True
LangSmith tracing: False


## Step 2 — Your first LangChain call

We'll instantiate a chat model, send a single user message, and inspect the structured response. The model class reads its API key (`OPENAI_API_KEY` / `GROQ_API_KEY`) from `os.environ` automatically — we don't pass the key in code.

Notes on the parameters:

- `temperature=0` — deterministic, repeatable output.
- `max_tokens=128` — output ceiling; keeps test calls cheap.

The return type is an `AIMessage` (a Pydantic model), **not** a plain string. It carries `content`, `response_metadata` (token usage, finish reason, model name), and `id`.

> **Provider switch.** The cell below is controlled by `USE_OPENAI`:
>
> - `USE_OPENAI = True` → `ChatOpenAI(model="gpt-4o-mini")`. Requires billing set up on your OpenAI account. If you see `RateLimitError: 429 insufficient_quota`, add a payment method at <https://platform.openai.com/account/billing>.
> - `USE_OPENAI = False` → `ChatGroq(model="llama-3.1-8b-instant")`. Works on Groq's free tier — no billing needed.
>
> All later cells reuse the `llm` variable, so the rest of the notebook is provider-agnostic. This is the LangChain abstraction win in practice: one line decides which company runs your inference.

In [3]:
USE_OPENAI = False

if USE_OPENAI:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=128)
else:
    from langchain_groq import ChatGroq
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_tokens=128)

print(f"Using: {type(llm).__name__} ({llm.model_name if hasattr(llm, 'model_name') else llm.model})")

response = llm.invoke("In one sentence, what is LangChain?")

print()
print("type        :", type(response).__name__)
print("content     :", response.content)
print("model       :", response.response_metadata.get("model_name") or response.response_metadata.get("model"))
print("finish      :", response.response_metadata.get("finish_reason"))
print("token usage :", response.usage_metadata)

Using: ChatGroq (llama-3.1-8b-instant)

type        : AIMessage
content     : LangChain is an open-source Python library that enables developers to build large language models (LLMs) into production-ready applications, providing a framework for integrating LLMs with external data and systems.
model       : llama-3.1-8b-instant
finish      : stop
token usage : {'input_tokens': 44, 'output_tokens': 40, 'total_tokens': 84}


## Step 3 — Multi-message conversation (system + human)

Real chat APIs accept a **list of messages**, each with a role (`system`, `human`, `ai`, `tool`). LangChain wraps these in typed message classes so your code is provider-agnostic.

- `SystemMessage` — sets the model's persona / rules. Sent once at the top.
- `HumanMessage` — a user turn.
- `AIMessage` — an assistant turn (e.g. when you replay history back to the model).

In [4]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a senior backend engineer. Answer in <= 2 sentences, no fluff."),
    HumanMessage(content="When should I use Redis vs Postgres for session storage in a high-traffic API?"),
]

response = llm.invoke(messages)
print(response.content)

Use Redis for session storage in a high-traffic API when you need fast, in-memory caching and can tolerate session loss in case of a Redis restart or failure. Use Postgres when you require persistent session storage and can't afford to lose session data, such as in a banking or e-commerce application.


## Step 4 — Your first LCEL chain: `prompt | model | parser`

This is the canonical LangChain pattern. We:

1. Define a **prompt template** with named variables (`{role}`, `{topic}`).
2. Pipe it into the **model**.
3. Pipe the model output into a **parser** that extracts plain `str` from the `AIMessage`.

The result is a single `Runnable` you can `.invoke()`, `.batch()`, or `.stream()`. The same chain can serve a single request, a batch of 1000, or a streaming HTTP response — without changing the chain definition.

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Be concise and technically precise."),
    ("human", "Explain {topic} in 2 sentences."),
])

chain = prompt | llm | StrOutputParser()

result = chain.invoke({
    "role": "principal backend engineer",
    "topic": "the difference between optimistic and pessimistic locking",
})
print(result)

Optimistic locking involves checking for concurrent modifications by verifying a version number or timestamp before updating a resource, and rolling back the update if the version number has changed, whereas pessimistic locking acquires a lock on the resource before updating it, preventing other transactions from modifying it until the lock is released. This approach ensures data consistency and prevents lost updates in optimistic locking, but may lead to deadlocks and increased contention in pessimistic locking.


### Bonus — `.batch()` and `.stream()` for free

The same chain object supports parallel batching and token-level streaming with zero extra code. This is the real payoff of LCEL.

In [6]:
batch_inputs = [
    {"role": "DBA", "topic": "B-tree vs hash indexes"},
    {"role": "SRE", "topic": "p99 latency vs average latency"},
    {"role": "security engineer", "topic": "JWT vs opaque session tokens"},
]

answers = chain.batch(batch_inputs)
for q, a in zip(batch_inputs, answers):
    print(f"Q ({q['role']}): {q['topic']}")
    print(f"A: {a}\n")

Q (DBA): B-tree vs hash indexes
A: B-tree indexes are self-balancing, multi-level index structures that use a combination of keys and pointers to efficiently locate data, making them suitable for range queries and ordered data, while hash indexes use a hash function to map keys to a specific location, making them ideal for equality queries and high-cardinality columns.

Q (SRE): p99 latency vs average latency
A: P99 latency (99th percentile latency) represents the latency value that 99% of requests fall below, providing insight into the worst-case scenario latency experienced by users, whereas average latency is a more general metric that can be skewed by outliers and may not accurately represent the typical user experience. A low p99 latency is more indicative of a system's responsiveness and reliability, as it ensures that most users are not impacted by long latency spikes.

Q (security engineer): JWT vs opaque session tokens
A: JSON Web Tokens (JWT) are digitally signed tokens that 

In [7]:
for chunk in chain.stream({
    "role": "staff engineer",
    "topic": "why N+1 queries kill API throughput",
}):
    print(chunk, end="", flush=True)
print()

N+1 queries occur when a database query is executed for each item in a collection, resulting in a linear increase in the number of database calls, which can lead to a significant degradation in API throughput due to increased latency and resource utilization. This is because each additional query incurs the overhead of network communication, database processing, and connection establishment, ultimately bottlenecking the system and limiting its ability to handle a large volume of requests.


In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

response = llm.invoke("What is LangChain? give me a short answer")

print(response.content)
print()
print("model :", llm.model)
print("usage :", response.usage_metadata)


LangChain is a framework that helps developers build applications powered by large language models (LLMs) by providing tools and abstractions to connect LLMs with other data sources and computation.

model : gemini-2.5-flash-lite
usage : {'input_tokens': 11, 'output_tokens': 35, 'total_tokens': 46, 'input_token_details': {'cache_read': 0}}
